# Cell Line Preprocessing

- **Purpose:** Interactive preprocessing of SCP542: keep colorectal cell lines, map genes to the shared table, rename cell lines, drop cell lines with fewer than 50 cells, and attach MSI and batch metadata.
- **Inputs:** SCP542 counts, `data/raw/geneAnno.csv`, `data/processed/genes_combined.csv`.
- **Outputs:** `data/processed/preprocessed_cell_line.h5ad` (written by `scripts/02_cell_line_pp.py`; this notebook does not save).
- **Result:** Ten CRC cell lines in the analyzed object, with `batch` set to the sequencing pool.


## Overview
This notebook preprocesses the cell line dataset SCP542 in preperation for CRC cancer cell line analysis. This notebook does the following:

1. Load the data with the metadata
2. Subset to combined genes + Add gene_id
3. Rename the cell line names
4. Remove cell lines with less than 50 cells
5. Add the MSI status manually (from cellosaurus)
   - Also add `batch` column for batch effect correction later on. In the cell line dataset it corresponds to the pool_id

In [1]:
import numpy as np
import pandas as pd
import anndata as ad
import os

from scipy import sparse

In [2]:
DATA_DIR = os.path.join("..", "..", "data")
PROCESSED_DIR = os.path.join(DATA_DIR, "processed")

gene_anno_file = os.path.join(DATA_DIR, "raw", "geneAnno.csv")
scp_counts_file = os.path.join(DATA_DIR, "raw", "SCP542_UMIcount_data.txt")
gene_file = os.path.join(PROCESSED_DIR, "genes_combined.csv")

# Load the cell line data with main metadata

In [3]:
def load_large_csv_in_sparse_chunks(
    file_path, meta_df, cell_ids=None, chunk_size=1000, skiprows=2
):
    """Load selected cell columns without materializing the full dense matrix.

    The SCP542 file is genes x cells. Chunks are therefore joined horizontally
    after transposition, producing a cells x genes CSR matrix.
    """
    metadata_cell_ids = meta_df.columns.astype(str).tolist()
    requested_cell_ids = metadata_cell_ids if cell_ids is None else [str(x) for x in cell_ids]

    header = pd.read_csv(file_path, sep="\t", skiprows=skiprows, nrows=0)
    expected_width = len(metadata_cell_ids) + 1
    if len(header.columns) != expected_width:
        raise ValueError(
            f"Count matrix has {len(header.columns) - 1} cell columns, "
            f"but metadata has {len(metadata_cell_ids)}"
        )

    metadata_cell_id_set = set(metadata_cell_ids)
    missing_cell_ids = sorted(set(requested_cell_ids) - metadata_cell_id_set)
    if missing_cell_ids:
        raise ValueError(
            f"Requested cell IDs are absent from the metadata: {missing_cell_ids[:5]}"
        )

    # The third line is a metadata row (Pool_ID), not the true cell header. Use
    # column positions and restore the cell IDs from the first-line metadata.
    requested_cell_set = set(requested_cell_ids)
    selected_cell_ids = [
        cell_id for cell_id in metadata_cell_ids if cell_id in requested_cell_set
    ]
    metadata_positions = {
        cell_id: position + 1 for position, cell_id in enumerate(metadata_cell_ids)
    }
    selected_positions = [metadata_positions[cell_id] for cell_id in selected_cell_ids]
    if not selected_cell_ids:
        raise ValueError("No requested cell IDs were found in the count matrix")

    chunks = []
    gene_symbols = []
    chunk_iter = pd.read_csv(
        file_path,
        sep="\t",
        index_col=0,
        skiprows=skiprows,
        usecols=[0] + selected_positions,
        chunksize=chunk_size,
    )

    print(
        f"    Loading {len(selected_cell_ids)} selected cell columns in sparse chunks..."
    )
    for i, chunk in enumerate(chunk_iter, start=1):
        chunk.index = chunk.index.astype(str)
        chunk.columns = selected_cell_ids
        gene_symbols.extend(chunk.index.tolist())
        counts_chunk = chunk.astype("uint32", copy=False)
        chunks.append(sparse.csr_matrix(counts_chunk.to_numpy(copy=False)).T)
        if i % 25 == 0:
            print(f"    Processed {i} gene chunks...")

    if not chunks:
        raise ValueError("The count matrix contains no gene rows")

    counts = sparse.hstack(chunks, format="csr", dtype=np.uint32)
    return counts, np.asarray(gene_symbols), selected_cell_ids

In [4]:
def load_cell_line_csv_as_adata(file_path, cell_line_contains=None):
    metadata = pd.read_csv(file_path, sep="\t", index_col=0, nrows=2)

    all_cell_ids = metadata.columns.astype(str).tolist()
    if cell_line_contains is not None:
        cell_lines = metadata.loc["Cell_line"].astype(str)
        selected_cell_ids = cell_lines.index[
            cell_lines.str.contains(cell_line_contains, regex=False, na=False)
        ].tolist()
    else:
        selected_cell_ids = all_cell_ids

    counts, gene_symbols, selected_cell_ids = load_large_csv_in_sparse_chunks(
        file_path,
        metadata,
        cell_ids=selected_cell_ids,
        chunk_size=500,
        skiprows=2,
    )

    adata = ad.AnnData(
        X=counts,
        obs=pd.DataFrame(index=selected_cell_ids),
        var=pd.DataFrame(index=gene_symbols),
    )

    adata.var["gene_symbol"] = adata.var_names
    adata.obs["cell_id"] = adata.obs_names
    selected_metadata = metadata.loc[:, selected_cell_ids]
    adata.obs["pool_id"] = selected_metadata.loc["Pool_ID"].to_numpy()
    adata.obs["cell_line"] = selected_metadata.loc["Cell_line"].to_numpy()
    adata.obs["chemistry"] = "v2"

    return adata

In [5]:
adata = load_cell_line_csv_as_adata(scp_counts_file, "LARGE_INTESTINE")

    Loading 2256 selected cell columns in sparse chunks...
    Processed 25 gene chunks...
    Processed 50 gene chunks...


In [6]:
adata

AnnData object with n_obs × n_vars = 2256 × 30314
    obs: 'cell_id', 'pool_id', 'cell_line', 'chemistry'
    var: 'gene_symbol'

In [7]:
print(adata.obs_names.is_unique, adata.var_names.is_unique)

True True


In [8]:
adata.obs.head(3)

,cell_id,pool_id,cell_line,chemistry
AATCCAGGTTTAGGAA-1-18,AATCCAGGTTTAGGAA-1-18,18,NCIH747_LARGE_INTESTINE,v2
ACATCAGTCATGGTCA-1-18,ACATCAGTCATGGTCA-1-18,18,NCIH747_LARGE_INTESTINE,v2
AGACGTTGTCCATCCT-1-18,AGACGTTGTCCATCCT-1-18,18,SW1116_LARGE_INTESTINE,v2


In [9]:
adata.var.head(3)

,gene_symbol
A1BG,A1BG
A1BG-AS1,A1BG-AS1
A1CF,A1CF


# Subset to combined genes from `Non_usable_gene.ipynb`

In [10]:
gene_df = pd.read_csv(gene_file)
gene_df = gene_df.set_index("gene_symbol")

print(
    f"Sanity check: All intersection genes in cell line data? \
    {gene_df.index.isin(adata.var_names).all()}"
)

Sanity check: All intersection genes in cell line data?     True


In [11]:
adata.var["gene_id"] = adata.var["gene_symbol"].map(gene_df["gene_id"])
gene_mask = adata.var["gene_id"].notna()
adata = adata[:, gene_mask].copy()

In [12]:
adata.var.head(3)

,gene_symbol,gene_id
A1BG,A1BG,ENSG00000121410
A1BG-AS1,A1BG-AS1,ENSG00000268895
A1CF,A1CF,ENSG00000148584


# Rename cell line names

In [13]:
adata.obs["cell_line"].value_counts().to_frame().style

,count
cell_line,
SNUC4_LARGE_INTESTINE,471
CL34_LARGE_INTESTINE,418
LS180_LARGE_INTESTINE,362
HT55_LARGE_INTESTINE,211
LS1034_LARGE_INTESTINE,153
RCM1_LARGE_INTESTINE,131
CCK81_LARGE_INTESTINE,127
OUMS23_LARGE_INTESTINE,107
HCC56_LARGE_INTESTINE,102


In [14]:
adata.obs["cell_line"] = adata.obs["cell_line"].str.split("_").str[0]

In [15]:
print(f"Cell line: {adata.shape[0]} cells x {adata.shape[1]} genes")
print(f"Number of cell lines: {adata.obs['cell_line'].nunique()}")
print("\n", adata.obs['cell_line'].value_counts())
print("\n", adata.obs['pool_id'].value_counts())

Cell line: 2256 cells x 28617 genes
Number of cell lines: 13

 cell_line
SNUC4      471
CL34       418
LS180      362
HT55       211
LS1034     153
RCM1       131
CCK81      127
OUMS23     107
HCC56      102
NCIH747     80
SNU81       49
SW1116      23
SW1463      22
Name: count, dtype: int64

 pool_id
15    782
9     471
10    418
22    284
16    198
18    103
Name: count, dtype: int64


# Remove cell ines with less than 50 cells

In [16]:
cell_line_counts = adata.obs["cell_line"].value_counts()
valid_cell_lines = cell_line_counts[cell_line_counts >= 50].index
adata = adata[adata.obs["cell_line"].isin(valid_cell_lines)].copy()

In [17]:
print(f"Cell line: {adata.shape[0]} cells x {adata.shape[1]} genes")
print(f"Number of cell lines: {adata.obs['cell_line'].nunique()}")
print("\n", adata.obs['cell_line'].value_counts())
print("\n", adata.obs['pool_id'].value_counts())

Cell line: 2162 cells x 28617 genes
Number of cell lines: 10

 cell_line
SNUC4      471
CL34       418
LS180      362
HT55       211
LS1034     153
RCM1       131
CCK81      127
OUMS23     107
HCC56      102
NCIH747     80
Name: count, dtype: int64

 pool_id
15    782
9     471
10    418
22    284
16    127
18     80
Name: count, dtype: int64


In [18]:
adata

AnnData object with n_obs × n_vars = 2162 × 28617
    obs: 'cell_id', 'pool_id', 'cell_line', 'chemistry'
    var: 'gene_symbol', 'gene_id'

# Add MSI status from cellosaurs
The cellosaurus statuses match our derived statuses. We also append the batch_key batch that we will utilize later.

In [19]:
msi_cellosaurus = {
    "SNUC4": True,
    "CL34": True, # both: Sanger is MSS; 3 Studies say MSI
    "LS180": True, # MSI-high
    "HT55": False,
    "LS1034": False,
    "RCM1": False,
    "CCK81": True, # MSI-high, cell line: Metastatic; Lymph node
    "OUMS23": False, #  cell line: Metastatic; Pericardial effusion
    "HCC56": False, #  Metastatic; Ascites, problematic cell line: Contaminated. Shown to be a HSC-57 derivative
    "NCIH747": False, # both: one study MSS; Sanger and one study say MSI-low, metastatic; Lymph node
    "SNU81": False,
    "SW1116": False,
    "SW1463": False
}

adata.obs["mmr_status"] = adata.obs["cell_line"].map(msi_cellosaurus)
adata.obs["mmr_status"] = adata.obs["mmr_status"].map({
    True: "MMRd",
    False: "MMRp"
})

In [20]:
adata.obs["batch"] = adata.obs["pool_id"].copy()
adata.obs["dataset"] = "cell_line"
adata.obs["label"] = "Unknown"
adata.obs["donor_id"] = "cell_line"

In [21]:
adata.obs["mmr_status"].isna().sum()

np.int64(0)

In [22]:
print(f"is view?: {adata.is_view}")
print(f"var index unique?: {adata.var.index.is_unique}")
print(f"obs index unique?: {adata.obs.index.is_unique}\n")
print(adata.obs.dtypes)
print("\n", adata.var.dtypes)
print(f"\n X type: {adata.X.dtype}")

is view?: False
var index unique?: True
obs index unique?: True

cell_id       object
pool_id       object
cell_line     object
chemistry     object
mmr_status    object
batch         object
dataset       object
label         object
donor_id      object
dtype: object

 gene_symbol    object
gene_id        object
dtype: object

 X type: uint32


In [23]:
cols = adata.obs.select_dtypes(include=["object"]).columns.tolist()
cols.remove("cell_id")
for col in cols:
    adata.obs[col] = adata.obs[col].astype("category")

In [24]:
print(f"is view?: {adata.is_view}")
print(f"var index unique?: {adata.var.index.is_unique}")
print(f"obs index unique?: {adata.obs.index.is_unique}\n")
print(adata.obs.dtypes)
print("\n", adata.var.dtypes)
print(f"\n X type: {adata.X.dtype}")

is view?: False
var index unique?: True
obs index unique?: True

cell_id         object
pool_id       category
cell_line     category
chemistry     category
mmr_status    category
batch         category
dataset       category
label         category
donor_id      category
dtype: object

 gene_symbol    object
gene_id        object
dtype: object

 X type: uint32


In [25]:
adata.obs.head(3)

,cell_id,pool_id,cell_line,chemistry,mmr_status,batch,dataset,label,donor_id
AATCCAGGTTTAGGAA-1-18,AATCCAGGTTTAGGAA-1-18,18,NCIH747,v2,MMRp,18,cell_line,Unknown,cell_line
ACATCAGTCATGGTCA-1-18,ACATCAGTCATGGTCA-1-18,18,NCIH747,v2,MMRp,18,cell_line,Unknown,cell_line
ATCATGGTCGCATGAT-1-18,ATCATGGTCGCATGAT-1-18,18,NCIH747,v2,MMRp,18,cell_line,Unknown,cell_line
